# Aspect 1 — Does Message Direction Matter in Relational GNNs?

Relational databases encode FK→PK relationships that give edges a natural direction (child → parent).
We ask: **does message-passing direction affect node classification on relational graphs?**

We test three modes across two architectures and two real-world datasets:

| Mode | Description |
|------|-------------|
| **MPNN-U** | Undirected — messages flow on all edges (FK→PK + reversed PK→FK) |
| **MPNN-D** | Directed — messages flow only on FK→PK edges (child → parent) |
| **Dir-GNN** | Bidirectional with separate parameters per direction |

**Hypothesis:** Since both tasks are user-level, signal should aggregate naturally upward via FK→PK edges (child rows → user). MPNN-D may match or exceed MPNN-U by avoiding reverse noise; Dir-GNN may do best by separating the two directions.

## 1. Setup

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import torch

# Resolve ROOT to aspect1/ regardless of where the kernel was started
_here = Path(os.path.abspath(""))
ROOT  = _here if _here.name == "aspect1" else _here / "aspect1"
os.chdir(ROOT)  # keep CWD in sync so relative subprocess calls work

PROCESSED   = ROOT / "processed"
CHECKPOINTS = ROOT / "checkpoints"
RESULTS_CSV = ROOT / "results" / "metrics.csv"
PYTHON      = sys.executable

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"ROOT:   {ROOT}")
print(f"Device: {DEVICE}")
print(f"Python: {PYTHON}")

# ── RESULTS_ONLY: skip preprocessing & training, jump straight to results ──────
RESULTS_ONLY = False

# ── FORCE_RERUN: delete results CSV and re-run all experiments from scratch ────
# Useful when you want to regenerate all results (e.g. after changing training code).
# Has no effect when RESULTS_ONLY=True.
FORCE_RERUN = False

## 2. Datasets

We use four [RelBench](https://relbench.stanford.edu/) datasets. The first two target a **parent (PK-side) node**; the third targets a **child (FK-side) node** — flipping the expected signal direction. The fourth is a **multiclass citation graph** to test directionality on a different domain.

### rel-stack / user-engagement  *(PK-side target)*
**Task:** Predict whether a user will receive a positive vote in the next 2 years.  
**Target:** `users` — parent node. Child rows (posts, comments, votes) have FK→user.  
**Expected:** MPNN-D competitive — signal flows FK→PK (child activity → user).

### rel-avito / user-visits  *(PK-side target)*
**Task:** Predict whether a user will visit an ad in the next 2 weeks.  
**Target:** `UserInfo` — parent node. Activity tables have FK→UserInfo.  
**Expected:** Same as rel-stack — MPNN-D should be competitive.

### rel-stack / post-votes  *(FK-side target)*
**Task:** Predict whether a post will receive at least one upvote (binarized vote count > 0; ~6.4% positive rate).  
**Target:** `posts` — child node. Posts have FK→users (post references its author).  
**Expected:** Signal should flow PK→FK (user quality → post). MPNN-D (FK→PK only) should *hurt* — it cuts off the direction carrying the relevant signal.

### rel-arxiv / author-category  *(multiclass, citation graph)*
**Task:** Predict the primary ArXiv subject category of an author (53 classes: cs.LG, math.CO, etc.).  
**Target:** `authors` — stable entity connected to papers, which link to other papers via citations.  
**Metric:** Macro-F1 (53-class multiclass).  
**Expected:** Dir-GNN may benefit from directed citation structure — citations flow from newer to older papers, and separating "citing" from "cited-by" contexts encodes different semantic signals.

In [ ]:
TASKS = [
    ("rel-stack",  "user-engagement"),
    ("rel-avito",  "user-visits"),
    ("rel-stack",  "post-votes"),
    ("rel-arxiv",  "author-category"),
]

for dataset, task in TASKS:
    meta_path = PROCESSED / dataset / task / "meta.json"
    if meta_path.exists():
        with open(meta_path) as f:
            meta = json.load(f)
        fwd = len(meta["fwd_edge_types"])
        rev = len(meta["rev_edge_types"])
        ntypes = len(meta["node_types"])
        feat_dims = meta["node_feat_dims"]
        task_type = meta.get("task_type", "binary")
        num_classes = meta.get("num_classes", 1)
        print(f"\n{dataset}/{task}  [{task_type}, {num_classes} class(es)]")
        print(f"  Target node : {meta['target_node']}")
        print(f"  Node types  : {ntypes}  ({', '.join(meta['node_types'])})")
        print(f"  Edges       : {fwd} FK→PK + {rev} rev PK→FK = {fwd+rev} total")
        print(f"  Feature dims: {feat_dims}")
    else:
        print(f"{dataset}/{task}: not yet preprocessed")

## 3. Preprocessing

The preprocessing script (`preprocess.py`) builds `train.pt`, `val.pt`, `test.pt`, and `meta.json`
for each dataset/task. It:
- Downloads the RelBench dataset via `relbench.get_dataset()` (cached locally after first run)
- Applies temporal splits using `db.upto(cutoff)` — train/val share the same cutoff timestamp
- Encodes numerical features with `StandardScaler`, categoricals with `OrdinalEncoder` (fitted on train only)
- Builds a `HeteroData` graph with FK→PK edges plus reversed PK→FK edges (prefixed `rev_`)
- Drops leakage columns from the task's `remove_columns` list

**If `.pt` files already exist, this cell is skipped.**

In [ ]:
if RESULTS_ONLY:
    print("RESULTS_ONLY=True — skipping preprocessing.")
else:
    def preprocessing_done():
        return all(
            (PROCESSED / ds / task / "meta.json").exists()
            for ds, task in TASKS
        )

    if preprocessing_done():
        print("Preprocessed data found — skipping preprocessing.")
    else:
        print("Running preprocess.py (this may take 20-40 minutes on first run) …")
        result = subprocess.run(
            [PYTHON, "preprocess.py"],
            cwd=ROOT,
            capture_output=False,
        )
        if result.returncode != 0:
            raise RuntimeError("preprocess.py failed — check output above.")
        print("\nPreprocessing complete.")

In [ ]:
# Show graph statistics from the preprocessed .pt files
for dataset, task in TASKS:
    meta_path = PROCESSED / dataset / task / "meta.json"
    if not meta_path.exists():
        continue
    with open(meta_path) as f:
        meta = json.load(f)
    target = meta["target_node"]
    is_multiclass = meta.get("task_type", "binary") == "multiclass"
    print(f"\n{'='*50}")
    print(f"{dataset} / {task}  [{meta.get('task_type','binary')}]")
    print(f"{'='*50}")
    for split in ["train", "val", "test"]:
        pt = PROCESSED / dataset / task / f"{split}.pt"
        hdata = torch.load(pt, weights_only=False, map_location="cpu")
        n_nodes = hdata[target].num_nodes
        n_labeled = int(hdata[target].mask.sum())
        if is_multiclass:
            n_classes = meta.get("num_classes", 1)
            print(f"  {split:5s}: {n_nodes:6,} nodes  {n_labeled:6,} labeled  ({n_classes} classes)")
        else:
            n_pos = int(hdata[target].y[hdata[target].mask].sum())
            print(f"  {split:5s}: {n_nodes:6,} nodes  {n_labeled:6,} labeled  "
                  f"{n_pos:5,} positive  ({100*n_pos/max(n_labeled,1):.1f}%)")
        del hdata

## 4. Model Architectures

All models share the same backbone:
- Lazy initialization (`(-1, -1)`) in SAGEConv/GATConv handles varying feature dimensions across node types
- `hidden_dim = 64`, `dropout = 0.3`
- Binary head: `Linear(64 → 1) + Sigmoid`
- `to_hetero()` lifts homogeneous convolutions to heterogeneous graphs

### MPNN-U (Undirected)
```
All edges (FK→PK ∪ rev PK→FK) → to_hetero(SAGEConv/GATConv) × L layers
```
Messages flow in both directions. Child rows push features to parent users, and vice versa.

### MPNN-D (Directed, FK→PK only)
```
FK→PK edges only → to_hetero(SAGEConv/GATConv) × L layers
```
Messages flow only from child rows up to parent nodes. Reverse edges are filtered out at forward-pass time.

### Dir-GNN (Bidirectional with Separate Parameters)
```
Per layer:
  fwd_out = to_hetero(Conv(hidden//2), fwd_edges)   # FK→PK
  rev_out = to_hetero(Conv(hidden//2), rev_edges)   # PK→FK
  x = Linear(cat([fwd_out, rev_out])) + ReLU        # project back to hidden
```
Each layer runs two separate convolutions (with independent parameters) then concatenates and projects. This is the full Dir-GNN formulation from Rossi et al. (2023).

In [ ]:
# Quick parameter count for each architecture × mode × layer combination
# (uses one dataset's metadata as a representative graph)
import sys; sys.path.insert(0, str(ROOT))
from models import build_model, count_parameters
from train import load_meta, load_split, make_loader, DEVICE as _DEV

_dataset, _task = "rel-stack", "user-engagement"
_meta = load_meta(_dataset, _task)
_data = load_split(_dataset, _task, "train")
_metadata = _data.metadata()
_target = _meta["target_node"]

print(f"{'Arch':6}  {'Mode':8}  {'L':2}  {'Params':>10}")
print("-" * 32)
for arch in ["sage", "gat"]:
    for mode in ["mpnn_u", "mpnn_d", "dir_gnn"]:
        for L in [1, 2, 3]:
            m = build_model(_metadata, _target, arch=arch, mode=mode,
                            num_layers=L, hidden_dim=64).to(_DEV)
            # warm-up lazy init
            _ldr = make_loader(_data, _target, "mask", 32, 1, L, False)
            _b = next(iter(_ldr)).to(_DEV)
            import torch
            with torch.no_grad():
                m({nt: _b[nt].x for nt in _b.node_types}, _b.edge_index_dict)
            n = count_parameters(m)
            print(f"{arch:6}  {mode:8}  {L:2}  {n:10,}")
    print()

## 5. Experiment Configuration

We run **72 experiments**: 4 datasets × 2 architectures × 3 modes × 3 depth levels × 1 seed.

| Axis | Values |
|------|--------|
| Dataset/Task | rel-stack/user-engagement, rel-avito/user-visits, rel-stack/post-votes, rel-arxiv/author-category |
| Architecture | GraphSAGE, GAT |
| Mode | MPNN-U, MPNN-D, Dir-GNN |
| Layers | 1, 2, 3 |
| Seed | 0 |

Training details:
- Adam optimizer, lr=1e-3, weight_decay=1e-5
- Max 100 epochs with **early stopping** (patience=10) on val AUPRC (binary) or val accuracy (multiclass)
- Batch size 512 with NeighborLoader (10 neighbors per layer)
- Binary tasks: **AUPRC** (area under precision-recall curve)
- Multiclass tasks (rel-arxiv): **Macro-F1** across 53 ArXiv categories

Checkpoints are saved to `checkpoints/{dataset}/{task}/{arch}_{mode}_L{layers}_s{seed}.pt`.
**Already-completed runs are skipped automatically.**

In [ ]:
ARCHS  = ["sage", "gat"]
MODES  = ["mpnn_u", "mpnn_d", "dir_gnn"]
LAYERS = [1, 2, 3]
SEED   = 0

def ckpt_path(dataset, task, arch, mode, num_layers, seed):
    return CHECKPOINTS / dataset / task / f"{arch}_{mode}_L{num_layers}_s{seed}.pt"

configs = [
    dict(dataset=ds, task=task, arch=arch, mode=mode, num_layers=L, seed=SEED)
    for ds, task in TASKS
    for arch in ARCHS
    for mode in MODES
    for L in LAYERS
]

done    = [c for c in configs if ckpt_path(**c).exists()]
pending = [c for c in configs if not ckpt_path(**c).exists()]
print(f"Total: {len(configs)}   Done: {len(done)}   Pending: {len(pending)}")
if pending:
    print("\nPending:")
    for c in pending:
        print(f"  {c['dataset']}/{c['task']}  {c['arch']}  {c['mode']}  L={c['num_layers']}")

## 6. Training

The cell below runs all pending configurations. Completed ones are automatically skipped unless `FORCE_RERUN=True`.

In [ ]:
if RESULTS_ONLY:
    print("RESULTS_ONLY=True — skipping training.")
else:
    if FORCE_RERUN and RESULTS_CSV.exists():
        import shutil
        backup = RESULTS_CSV.with_suffix(".csv.bak")
        shutil.copy(RESULTS_CSV, backup)
        RESULTS_CSV.unlink()
        print(f"FORCE_RERUN=True — backed up and deleted {RESULTS_CSV.name}, re-running all configs.")

    cmd = [PYTHON, "-u", "run_experiments.py"]
    if FORCE_RERUN:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=ROOT, capture_output=False)
    if result.returncode != 0:
        raise RuntimeError("run_experiments.py failed — check output above.")

    # Re-check
    done    = [c for c in configs if ckpt_path(**c).exists()]
    pending = [c for c in configs if not ckpt_path(**c).exists()]
    print(f"\nAfter training: {len(done)}/{len(configs)} complete")
    if pending:
        print("Still pending (may have failed):")
        for c in pending:
            print(f"  {c['dataset']}/{c['task']}  {c['arch']}  {c['mode']}  L={c['num_layers']}")

## 7. Results

We load metrics from `results/metrics.csv` — one row per completed run — and analyze:
1. Main comparison: mode (MPNN-U vs MPNN-D vs Dir-GNN) by dataset and architecture
2. Effect of depth (L=1,2,3) per mode
3. Test AUPRC heatmap

In [ ]:
if not RESULTS_CSV.exists():
    raise FileNotFoundError(f"No results file at {RESULTS_CSV}. Run training first.")

df = pd.read_csv(RESULTS_CSV)
print(f"Loaded {len(df)} rows from metrics.csv")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
df_all = df.drop_duplicates(
    subset=["dataset", "task", "mode", "arch", "num_layers", "seed"],
    keep="last"
).reset_index(drop=True)

MODE_LABELS = {"mpnn_u": "MPNN-U", "mpnn_d": "MPNN-D", "dir_gnn": "Dir-GNN"}
df_all["Mode"]    = df_all["mode"].map(MODE_LABELS)
df_all["Arch"]    = df_all["arch"].str.upper()
df_all["Dataset"] = df_all["dataset"] + "/" + df_all["task"]

# Unified metric: macro_f1 for multiclass (rel-arxiv), AUPRC for binary
def primary_metric(row):
    f1 = row.get("test_macro_f1")
    if pd.notna(f1) and f1 > 0:
        return f1
    return row.get("test_auprc", float("nan"))

df_all["test_metric"] = df_all.apply(primary_metric, axis=1)
df_all["metric_label"] = df_all.apply(
    lambda r: "Macro-F1" if pd.notna(r.get("test_macro_f1")) and r.get("test_macro_f1", 0) > 0
              else "AUPRC", axis=1
)

print(f"Experiment rows: {len(df_all)}  (expecting up to 72)")
print(df_all.groupby(["Dataset", "Arch", "Mode"])["test_metric"].mean().round(4).to_string())

# Shared variables for all figures
modes       = ["MPNN-U", "MPNN-D", "Dir-GNN"]
archs       = ["SAGE", "GAT"]
datasets    = sorted(df_all["Dataset"].unique())
colors      = ["#4C72B0", "#DD8452", "#55A868"]
arch_colors = {"SAGE": "#4C72B0", "GAT": "#DD8452"}


In [ ]:
# ── Figure 1: Effect of Depth (layers) per Mode ───────────────────────────────
fig, axes = plt.subplots(1, len(datasets), figsize=(6*len(datasets), 5))
layer_vals  = [1, 2, 3]
mode_colors = dict(zip(modes, colors))

for ax, dataset in zip(axes, datasets):
    sub = df_all[df_all["Dataset"] == dataset]
    ylabel = sub["metric_label"].iloc[0] if len(sub) else "AUPRC"
    for mode in modes:
        mode_sub = sub[sub["Mode"] == mode]
        ys = [
            mode_sub[mode_sub["num_layers"] == L]["test_metric"].mean()
            for L in layer_vals
        ]
        ax.plot(layer_vals, ys, marker="o", label=mode,
                color=mode_colors[mode], linewidth=2)
    ax.set_xticks(layer_vals)
    ax.set_xlabel("Number of Layers")
    ax.set_ylabel(f"Test {ylabel} (avg over both archs)")
    ax.set_title(dataset, fontsize=11, fontweight="bold")
    ax.legend()
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Figure 1: Effect of Depth on Test Metric", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(ROOT / "results" / "fig1_depth_effect.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Figure 2: Heatmap — Mode × Dataset, avg test metric ───────────────────────
pivot = df_all.groupby(["Dataset", "Mode"])["test_metric"].mean().unstack("Mode")
pivot = pivot[["MPNN-U", "MPNN-D", "Dir-GNN"]]

fig, ax = plt.subplots(figsize=(7, 3))
im = ax.imshow(pivot.values, aspect="auto", cmap="YlGn")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, fontsize=11)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=10)
plt.colorbar(im, ax=ax, label="Avg Test Metric")
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.3f}", ha="center", va="center",
                    color="black", fontsize=10, fontweight="bold")
ax.set_title("Figure 2: Avg Test Metric by Mode (AUPRC / Macro-F1)", fontsize=11)
plt.tight_layout()
plt.savefig(ROOT / "results" / "fig2_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Figure 3: Parameter Efficiency — Metric vs Param Count ────────────────────
fig, axes = plt.subplots(1, len(datasets), figsize=(9 * len(datasets), 6), sharey=False)
mode_colors  = {"MPNN-U": "#4C72B0", "MPNN-D": "#DD8452", "Dir-GNN": "#55A868"}
arch_markers = {"SAGE": "o", "GAT": "^"}

for ax, dataset in zip(axes, datasets):
    sub = df_all[df_all["Dataset"] == dataset]
    ylabel = sub["metric_label"].iloc[0] if len(sub) else "AUPRC"
    for mode in modes:
        for arch in archs:
            pts = sub[(sub["Mode"] == mode) & (sub["Arch"] == arch)]
            if pts.empty:
                continue
            ax.scatter(
                pts["num_params"] / 1_000,
                pts["test_metric"],
                color=mode_colors[mode],
                marker=arch_markers[arch],
                s=100, alpha=0.85, zorder=3,
                label=f"{mode} / {arch}" if dataset == datasets[0] else "_",
            )
            for _, row in pts.iterrows():
                ax.annotate(
                    f"L{int(row['num_layers'])}",
                    (row["num_params"] / 1_000, row["test_metric"]),
                    textcoords="offset points", xytext=(6, 3), fontsize=8, alpha=0.85,
                )
    ax.set_xlabel("Parameter Count (k)", fontsize=11)
    ax.set_ylabel(f"Test {ylabel}", fontsize=11)
    ax.set_title(dataset, fontsize=12, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)

handles = [
    plt.scatter([], [], color=mode_colors[m], marker=arch_markers[a],
                s=70, label=f"{m} / {a}")
    for m in modes for a in archs
]
axes[0].legend(handles=handles, fontsize=9, title="Mode / Arch",
               loc="lower right", framealpha=0.8)

fig.suptitle("Figure 3: Parameter Efficiency (test metric vs param count)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(ROOT / "results" / "fig3_param_efficiency.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Table 1: Full results summary ─────────────────────────────────────────────
summary = (
    df_all.groupby(["Dataset", "Arch", "Mode", "num_layers"])
    [["test_metric", "train_time_sec"]]
    .mean()
    .round(4)
)
summary.columns = ["Test Metric (AUPRC / Macro-F1)", "Train Time (s)"]
print("Full Results Table")
print(summary.to_string())


In [ ]:
# ── Best configuration per dataset ────────────────────────────────────────────
print("Best configuration per dataset (by test metric):\n")
for dataset in datasets:
    sub = df_all[df_all["Dataset"] == dataset]
    if sub.empty:
        continue
    best = sub.loc[sub["test_metric"].idxmax()]
    metric_name = best["metric_label"]
    print(f"{dataset}")
    print(f"  Best:  {best['Arch']} / {best['Mode']} / L={best['num_layers']}")
    print(f"  Test {metric_name}: {best['test_metric']:.4f}")
    print()


## 8. Analysis & Conclusions

### What we tested
We evaluated three message-passing modes — **MPNN-U** (undirected), **MPNN-D** (FK→PK only), and **Dir-GNN** (bidirectional, separate parameters) — on four relational datasets using GraphSAGE and GAT backbones at depths 1, 2, and 3.

### Key findings

**Depth matters most for rel-stack:**  
Going from L=2 to L=3 nearly doubles AUPRC on both rel-stack tasks (post-votes: 0.14→0.33, user-engagement: 0.21→0.31). rel-avito is flat across all depths. This reflects how deep the relevant signal sits in each schema.

**Directionality matters when the target is structurally constrained:**  
- *rel-stack/post-votes* (FK-side target): Dir-GNN and MPNN-U beat MPNN-D at L=3 (~12% margin). MPNN-D cannot receive user features into posts because users are upstream in the FK→PK direction.  
- *rel-stack/user-engagement* and *rel-avito* (PK-side targets): modes nearly tied — FK→PK edges already carry the main signal, so reverse edges add little.  
- *rel-arxiv/author-category* (citation graph, multiclass): citations are directed (newer cites older); Dir-GNN has independent parameters per direction, which may help distinguish "citing context" from "cited-by context".

**Dir-GNN is inconsistent:**  
Despite having separate weights per direction, Dir-GNN shows high variance across datasets and rarely uniformly outperforms MPNN-U. The added complexity does not reliably translate to better performance.

### Summary
The hypothesis is confirmed on the datasets where the structural mismatch is clearest: MPNN-D underperforms when the target node cannot receive key information via forward edges alone. rel-arxiv/author-category adds a directed graph with multiclass output, testing whether separate directional parameters help in a citation network context.

## 9. Effective Parameters

Not all parameters in a k-layer GNN actually influence the output. The three modes differ structurally in which edge types are ever used:

- **MPNN-D**: only FK→PK (forward) edges are included in the model — parameters exist only for forward edges. An edge is "effective" if its destination node type can propagate to the target within the remaining layers.
- **MPNN-U**: all edges (fwd + rev), with the same L-hop reachability constraint.
- **Dir-GNN**: separate parameter matrices for fwd and rev edges. Effective fraction equals MPNN-U's (same reachability), but each direction has independent weights.

**Key structural difference for post-votes (FK-side target):** The target node `posts` has a FK→users relationship. MPNN-D only uses fwd edges (posts→users), so it can never receive messages *from* users into posts. MPNN-U uses the rev edge (users→posts) to bring user context into posts at L=1. This directly explains MPNN-D's performance gap on the FK-side task.

**rel-arxiv/author-category:** The citation graph is strongly directed — papers cite other papers forward in time. All 6 node types (authors, papers, paperAuthors, paperCategories, categories, citations) are reachable from `authors` in 1–2 hops via both fwd and rev edges, so the effective parameter fraction is high for all modes. The key question here is whether Dir-GNN's separate directional weights add expressiveness over MPNN-U on a naturally directed graph.

In [ ]:
# ── Effective Parameter Analysis: AUPRC vs Effective Param Count ──────────────
import json, os
from pathlib import Path
from collections import deque
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd

ROOT = Path(os.path.abspath(""))
if ROOT.name != "aspect1":
    ROOT = ROOT / "aspect1"

def bfs_backwards(target, active_edges):
    incoming = {}
    for src, e, dst in active_edges:
        incoming.setdefault(dst, set()).add(src)
    dist = {target: 0}
    q = deque([target])
    while q:
        n = q.popleft()
        for s in incoming.get(n, set()):
            if s not in dist:
                dist[s] = dist[n] + 1
                q.append(s)
    return dist

def eff_fraction(active_edges, dist, L):
    if not active_edges:
        return 1.0
    active = sum(
        1 for l in range(L) for _, _, dst in active_edges
        if dist.get(dst, float('inf')) <= L - 1 - l
    )
    return active / (len(active_edges) * L)

TASK_LIST = [
    ("rel-stack",  "user-engagement"),
    ("rel-avito",  "user-visits"),
    ("rel-stack",  "post-votes"),
    ("rel-arxiv",  "author-category"),
]

eff_frac_map = {}
for ds, task in TASK_LIST:
    meta_path = ROOT / "processed" / ds / task / "meta.json"
    if not meta_path.exists():
        continue
    with open(meta_path) as f:
        meta = json.load(f)
    target = meta["target_node"]
    fwd_et = [tuple(e) for e in meta["fwd_edge_types"]]
    all_et = fwd_et + [tuple(e) for e in meta["rev_edge_types"]]
    dist_d = bfs_backwards(target, fwd_et)
    dist_u = bfs_backwards(target, all_et)
    for L in [1, 2, 3]:
        eff_frac_map[(ds, task, "mpnn_d",  L)] = eff_fraction(fwd_et, dist_d, L)
        eff_frac_map[(ds, task, "mpnn_u",  L)] = eff_fraction(all_et, dist_u, L)
        eff_frac_map[(ds, task, "dir_gnn", L)] = eff_fraction(all_et, dist_u, L)

df = pd.read_csv(ROOT / "results" / "metrics.csv")
df = df.drop_duplicates(
    subset=["dataset", "task", "mode", "arch", "num_layers", "seed"], keep="last"
).reset_index(drop=True)
df["eff_frac"] = df.apply(
    lambda r: eff_frac_map.get((r["dataset"], r["task"], r["mode"], r["num_layers"]), 1.0), axis=1
)
df["eff_params"] = (df["num_params"] * df["eff_frac"]).round().astype(int)

# For binary tasks use AUPRC; for multiclass use macro_f1
df["metric"] = df.apply(
    lambda r: r["test_macro_f1"] if pd.notna(r.get("test_macro_f1")) and r.get("test_macro_f1", 0) > 0
              else r["test_auprc"], axis=1
)
df["metric_label"] = df.apply(
    lambda r: "Macro-F1" if pd.notna(r.get("test_macro_f1")) and r.get("test_macro_f1", 0) > 0
              else "AUPRC", axis=1
)

MODE_COLORS  = {"mpnn_u": "#4C72B0", "mpnn_d": "#DD8452", "dir_gnn": "#55A868"}
MODE_LABELS  = {"mpnn_u": "MPNN-U", "mpnn_d": "MPNN-D", "dir_gnn": "Dir-GNN"}
ARCH_MARKERS = {"sage": "o", "gat": "^"}

available = [(ds, t) for ds, t in TASK_LIST if (ROOT / "processed" / ds / t / "meta.json").exists()]
fig, axes = plt.subplots(1, len(available), figsize=(7 * len(available), 6), sharey=False)
if len(available) == 1:
    axes = [axes]
fig.suptitle(
    "Parameter Efficiency: Test Metric vs Effective Parameter Count\n"
    "(effective = params whose edge/node types lie within GNN reach of the target)",
    fontsize=12, fontweight="bold",
)

for ax, (ds, task) in zip(axes, available):
    sub = df[(df["dataset"] == ds) & (df["task"] == task)]
    ylabel = sub["metric_label"].iloc[0] if len(sub) else "AUPRC"
    for mode in ["mpnn_u", "mpnn_d", "dir_gnn"]:
        for arch in ["sage", "gat"]:
            pts = sub[(sub["mode"] == mode) & (sub["arch"] == arch)]
            if pts.empty:
                continue
            ax.scatter(pts["eff_params"] / 1000, pts["metric"],
                       c=MODE_COLORS[mode], marker=ARCH_MARKERS[arch],
                       s=100, alpha=0.85, edgecolors="white", linewidths=0.8, zorder=3)
            for _, row in pts.iterrows():
                ax.annotate(f"L={int(row['num_layers'])}",
                            (row["eff_params"] / 1000, row["metric"]),
                            textcoords="offset points", xytext=(4, 2), fontsize=7, alpha=0.85)
    ax.set_xlabel("Effective Parameter Count (k)", fontsize=11)
    ax.set_ylabel(f"Test {ylabel}", fontsize=11)
    ax.set_title(f"{ds} / {task}", fontsize=11, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.2)

legend_elements = [
    mpatches.Patch(color=MODE_COLORS[m], label=MODE_LABELS[m])
    for m in ["mpnn_u", "mpnn_d", "dir_gnn"]
] + [
    Line2D([0], [0], marker=ARCH_MARKERS[a], color="w",
           markerfacecolor="#555", markersize=9, label=a.upper())
    for a in ["sage", "gat"]
]
axes[-1].legend(handles=legend_elements, fontsize=9, title="Mode / Arch",
                loc="lower right", framealpha=0.8)
plt.tight_layout()
plt.savefig(ROOT / "results" / "fig_effective_params_scatter.png", dpi=150, bbox_inches="tight")
plt.show()